# common/splits — Bộ chia Train/Validation/Test và 5 fold

Notebook chứa hàm dùng chung, không tự train và không tự ghi kết quả. Các notebook chính nạp tự động bằng `%run`. Khi sửa hàm, khởi động lại kernel rồi Run All notebook chính.


## Nạp các hàm phụ thuộc
Cell này cũng cho phép mở và Run All riêng notebook dùng chung.

In [ ]:
from pathlib import Path
import os
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'common/data_io.ipynb').is_file()), None)
if root is None:
    raise FileNotFoundError('Mở notebook trong thư mục project đầy đủ.')
os.chdir(root)
get_ipython().run_line_magic('run', 'common/data_io.ipynb')


## Thư viện và hằng số

In [ ]:
import json
from pathlib import Path

import numpy as np

from sklearn.model_selection import StratifiedKFold, train_test_split


## `create_splits`
Chia Train/Validation/Test phân tầng và tạo 5 fold trên Train; giữ chỉ số dòng gốc (bắt đầu từ 0, không tính dòng tiêu đề).

In [ ]:
def create_splits(df, path, output, seed, test_size, validation_size):
    if not 0 < test_size < 1 or not 0 < validation_size < 1 or test_size + validation_size >= 1:
        raise ValueError('Test and validation proportions must be positive and sum to less than 1.')
    idx = np.arange(len(df))
    remain, test = train_test_split(idx, test_size=test_size, random_state=seed, stratify=df[TARGET])
    train, val = train_test_split(remain, test_size=validation_size/(1-test_size), random_state=seed, stratify=df.iloc[remain][TARGET])
    folds = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for fit, valid in cv.split(train, df.iloc[train][TARGET]):
        folds.append({'train': train[fit].tolist(), 'validation': train[valid].tolist()})
    result = {'sha256': fingerprint(path), 'n_rows': len(df), 'seed': seed, 'test_size': test_size,
              'validation_size': validation_size, 'train': train.tolist(), 'validation': val.tolist(),
              'test': test.tolist(), 'folds': folds}
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    Path(output).write_text(json.dumps(result, indent=2), encoding='utf-8')
    return result


## `load_splits`
Kiểm tra dữ liệu khớp bộ chia đã lưu, các tập không chồng lấn và phủ đủ mọi dòng. Từ chối chỉ số trùng trong mỗi tập và trong từng danh sách train/validation của mỗi fold; thông báo nêu vị trí và chỉ số bị trùng.

In [ ]:
def load_splits(path, data_path, df):
    from collections import Counter

    s = json.loads(Path(path).read_text(encoding='utf-8'))
    if s['sha256'] != fingerprint(data_path) or s['n_rows'] != len(df):
        raise ValueError('Dataset differs from the dataset used to create splits. Do not reuse these indices.')
    index_lists = [(name, s[name]) for name in ('train', 'validation', 'test')]
    for fold_number, fold in enumerate(s['folds'], start=1):
        index_lists.extend((f'fold {fold_number} {name}', fold[name])
                           for name in ('train', 'validation'))
    for location, indices in index_lists:
        duplicates = [index for index, count in Counter(indices).items() if count > 1]
        if duplicates:
            raise ValueError(f'Duplicate row indices in {location}: {duplicates}')
    parts = [set(s[k]) for k in ('train', 'validation', 'test')]
    if any(parts[i] & parts[j] for i in range(3) for j in range(i)) or set.union(*parts) != set(range(len(df))):
        raise ValueError('Invalid split indices.')
    return s
